In [ ]:
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
import requests
import os
import random
import json
import re
import datetime
from urllib.parse import unquote, quote, unquote, urljoin

def get_api_key():
    # The URL of your API endpoint
    api_url = "https://sibeux.my.id/cloud-music-player/database/mobile-music-player/api/gdrive_api"
    API_KEY = ""
    # Make the GET request to the API
    try:
        response = requests.get(api_url)
        # Raise an exception for HTTP errors (4xx or 5xx)
        response.raise_for_status()

        # Parse the JSON response
        data = response.json()

        # Assuming you want to find an API key associated with a Google email (gmail.com)
        # You can specify a more precise email if needed, e.g., "animenatsuchan@gmail.com"

        found_google_api_key = False
        random.shuffle(data)  # Shuffle the data to randomize the search
        for item in data:
            email = item.get("email")
            if email and "@gmail.com" in email:  # Check if email exists and contains "@gmail.com"
                API_KEY = item.get("gdrive_api")
                # print(f"Found Google API Key for {email}: {API_KEY}")
                found_google_api_key = True
                break  # Exit the loop once the first Google API key is found

        if not found_google_api_key:
            print("No Google API Key (email ending with @gmail.com) found in the response.")

    except requests.exceptions.RequestException as e:
        print(f"Error making API request: {e}")
    except json.JSONDecodeError as e:
        print(f"Error decoding JSON response: {e}")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
    return API_KEY

def extract_folder_id(drive_url):
    # Regular expression to extract the folder ID from the URL
    match = re.search(r'/folders/([a-zA-Z0-9_-]+)', drive_url)
    if match:
        return match.group(1)
    else:
        raise ValueError("Invalid Google Drive folder URL")

def generate_random_duration():
    # Durasi dalam detik, antara 30 detik dan 5 menit (300 detik)
    duration_seconds = random.randint(30, 300)
    minutes = duration_seconds // 60
    seconds = duration_seconds % 60
    return f"{minutes:02}:{seconds:02}"

def escape_sql_string(value):
    # Escape single quotes by doubling them
    return value.replace("'", "''")

def escape_ampersand(url):
    # Sebelumnya ada return, dan harus dihapus agar semua karakter bisa dicheck.
    # Ini sebenarnya gak dipakai.
    if "&" in url:
        url = url.replace("&", "%26")
    if "☆" in url:
        url = url.replace("☆", "%E2%98%86")
    if " " in url:
        url = url.replace(" ", "%20")
    if "⁇" in url:
        url = url.replace("⁇", "%E2%81%87")
    if "’" in url:
        url = url.replace("’", "%E2%80%99")
    if "⁚" in url:
        url = url.replace("⁚", "%E2%81%9A")
    if "⁄" in url:
        url = url.replace("⁄", "%E2%81%84")

    return url

def escape_url(url):
    # Mengencode karakter khusus lainnya menggunakan urllib
    return quote(url, safe=":/")

def get_artist_from_title(title, list_title_artist):
    """Fungsi untuk mencocokkan judul dan menemukan artis yang sesuai."""
    for item in list_title_artist:
        # Pisahkan berdasarkan ' --- '
        song_title, artist = item.split(' --- ')
        if title.lower() == song_title.lower():  # Cocokkan tanpa memperhatikan huruf besar/kecil
            return artist
    return "Unknown Artist"  # Return "Unknown Artist" string jika tidak ditemukan

def get_match_title_from_name(title, list_index_title):
    """Fungsi untuk mencocokkan index dan menemukan judul yang sesuai."""
    for item in list_index_title:
        # Pisahkan berdasarkan ' --- '
        song_title, artist = item.split(' --- ')
        if title.lower() == song_title.lower():  # Cocokkan tanpa memperhatikan huruf besar/kecil
            return artist
    return "Unknown Title"  # Return "Unknown Title" string jika tidak ditemukan

def fetch_gitrepo_files(repo_url):
    """
    Example URL Folder:
    Github: https://github.com/cybeat-music/mitsuboshi-colors/tree/main/COLORS%20POWER
    Bitbucker: https://bitbucket.org/sibeux/arabasta-boom/src/main/Lost_Eve/
    Codeberg: https://codeberg.org/sibeux/regalia-freeze/src/branch/main/KONTINUUM
    Gitlab: https://gitlab.com/habiqi/regarden-academy/-/tree/main/Watch%20me?ref_type=heads
    """

    # Ekstrak informasi pemilik, repository, dan path folder dari URL
    parts = repo_url.split("/")
    owner = parts[3] if "https://drive.google.com" not in repo_url else ''
    repo = parts[4] if "https://drive.google.com" not in repo_url else ''
    branch = parts[6] if "https://drive.google.com" not in repo_url else ''
    folder_path = "/".join(parts[7:]) if "https://drive.google.com" not in repo_url else ''
    music_path = []

    # Daftar pasangan nama lagu dan artis
    list_title_artist = []
    # Daftar pasangan nama lagu dan index
    list_index_title = []

    list_cover = []
    category = ""
    artist = ""
    album_artist = ""
    album = ''

    # Untuk mencari nama file
    file_name_key = ""
    # Untuk cek jenis file = direktori
    file_dir_value = ""
    # Untuk cek jenis file = file
    file_type_value = ""

    is_dynamic_artist = False
    is_dynamic_title = False
    disc = 0

    folder_api_url = ""

    if "github.com/" in repo_url:
        folder_api_url = f"https://api.github.com/repos/{owner}/{repo}/contents/{folder_path}"
    elif "https://bitbucket.org/" in repo_url:
        folder_api_url = f"https://api.bitbucket.org/2.0/repositories/{owner}/{repo}/src/{branch}/{folder_path}"
    elif "https://codeberg.org"  in repo_url:
        # Khusus untuk Codeberg & Gitlab, pakai index 8 karena link-nya lebih panjang
        folder_path = "/".join(parts[8:])
        folder_api_url = f"https://codeberg.org/api/v1/repos/{owner}/{repo}/contents/{folder_path}"
    elif "https://gitlab.com/" in repo_url:
        folder_path = parts[8].split("?")[0]  # Mengambil path sebelum query string
        folder_api_url = f"https://gitlab.com/api/v4/projects/{owner}%2F{repo}/repository/tree?path={folder_path}&ref=main"

    folder_response = None
    gdrive_files = None
    if "https://drive.google.com" not in repo_url:
        folder_response = requests.get(folder_api_url)
    else:
        try:
            folder_id = extract_folder_id(repo_url)
            service = build("drive", "v3", developerKey=get_api_key())

            # By default, GDRIVE mengurutkan dari waktu terbaru dulu.
            # Kita modifikasi untuk mengurutkan berdasarkan nama file => orderBy="name"
            query = f"'{folder_id}' in parents"
            response = service.files().list(q=query, orderBy="name",
                                        fields="files(id, name)").execute()
            gdrive_files = response.get("files", [])
            
            # Dummy response to simulate successful API call
            folder_response = requests.Response()
            folder_response.status_code = 200

            if not gdrive_files:
                print("No files found.")
                return
        except HttpError as error:
            print(f"An error occurred: {error}")
        except ValueError as error:
            print(f"An error occurred: {error}")

    if folder_response.status_code == 200:
        files = None
        if "https://drive.google.com" in repo_url:
            files = gdrive_files
        else:
            files = folder_response.json()
        # Tentukan jenis file berdasarkan URL repository.
        if "github.com/" in repo_url:
            file_name_key = "name"
            file_dir_value = "dir"
            file_type_value = "file"
        elif "https://bitbucket.org/" in repo_url:
            file_name_key = "path"
            file_dir_value = "commit_directory"
            file_type_value = "commit_file"
            files = files.get("values", [])
        elif "https://codeberg.org" in repo_url:
            file_name_key = "name"
            file_dir_value = "dir"
            file_type_value = "file"
        elif "https://gitlab.com/" in repo_url:
            file_name_key = "name"
            file_dir_value = "tree"
            file_type_value = "blob"
        elif "https://drive.google.com" in repo_url:
            file_name_key = "name"
        # Dapatkan ID atau nama folder lagunya.
        for file in files:
            if "https://drive.google.com" not in repo_url:
                if file["type"] == file_dir_value:
                    file_name_without_ext = os.path.splitext(file[file_name_key])[0]
                    if "https://bitbucket.org/" in repo_url:
                        file_name_without_ext = file[file_name_key].split("/")[-1]
                    if file_name_without_ext == "flac":
                        music_path.append(file["path"])
                    if file_name_without_ext.isdigit():
                        music_path.append(file["path"])
            else:
                if file[file_name_key] == "flac":
                    music_path.append(file["id"])
                # Cek apakah nama folder adalah angka.
                if file[file_name_key].isdigit():
                    music_path.append(file["id"])
            # Dapatkan file .here.txt dan ekstrak informasi dari file tersebut.
            if '.here.txt' in file[file_name_key]:
                txt_url = ""
                if "github.com/" in repo_url:
                    txt_url = file["download_url"]
                elif "https://bitbucket.org/" in repo_url:
                    txt_url = file['links']['self']['href']
                elif "https://codeberg.org" in repo_url:
                    txt_url = file['download_url']
                elif "https://gitlab.com/" in repo_url:
                    branch = parts[7]
                    txt_url = f"https://gitlab.com/{owner}/{repo}/-/raw/{branch}/{folder_path}/.here.txt"
                elif "https://drive.google.com" in repo_url:
                    # Untuk Google Drive, kita perlu menggunakan API untuk mendapatkan konten file
                    file_id = file['id']
                    txt_url = f"https://www.googleapis.com/drive/v3/files/{file_id}?alt=media&key={get_api_key()}"
                # Lakukan request untuk mendapatkan konten file .here.txt
                response = requests.get(txt_url)
                # Secara eksplisit set encoding ke 'utf-8'
                response.encoding = 'utf-8'
                text_content = json.loads(response.text)
                category = text_content["category"]
                album = text_content['title']
                album = album.replace("'", "`")
                artist = text_content["author"]
                album_artist = text_content["author"]
                if text_content['disc'] != '0':
                    disc = int(text_content['disc']) - 1
                    if (text_content['1']) != []:
                        is_dynamic_artist = True
                        for i in range(disc+1):
                            list_title_artist.append(
                                text_content[str(i+1)]
                            )
                elif text_content['1'] != []:
                    is_dynamic_artist = True
                    list_title_artist.append(text_content["1"])
                if text_content['music_name_index']['1'] != []:
                    is_dynamic_title = True
                    for i in range(disc+1):
                        list_index_title.append(
                            text_content['music_name_index'][str(i+1)]
                        )
            # Dapatkan cover image dari file yang ada di dalam folder.
            if "https://drive.google.com" not in repo_url:
                if file['type'] == file_type_value:
                    ext = os.path.splitext(file[file_name_key])[1].lower()
                    if ext in ['.jpg', '.jpeg', '.png']:
                        image_url = ""
                        if "github.com/" in repo_url:
                            image_url = file["download_url"]
                        elif "https://bitbucket.org/" in repo_url:
                            image_url = file['links']['self']['href']
                        elif "https://codeberg.org" in repo_url:
                            image_url = file['download_url']
                        elif "https://gitlab.com/" in repo_url:
                            image = escape_ampersand(file[file_name_key])
                            image_url = f"https://gitlab.com/{owner}/{repo}/-/raw/{branch}/{folder_path}/{image}"
                        list_cover.append(image_url)
            else:
                ext = file[file_name_key].split('.')[-1].lower()
                if ext in ['jpg', 'jpeg', 'png']:
                    # Untuk Google Drive, kita perlu menggunakan ID file untuk mendapatkan URL gambar
                    image_url = f"https://drive.google.com/file/d/{file['id']}/view?usp=drive_link"
                    list_cover.append(image_url)

    # Starting timestamp for date_added
    base_time = datetime.datetime.now().replace(microsecond=0)

    # Base SQL INSERT statement template
    insert_statement_template = "INSERT INTO music (id_music, category, link_gdrive, title, artist, album, time, cover, favorite, date_added) VALUES \n"
    playlist_insert_statement_template = "INSERT INTO `playlist` (`uid`, `name`, `image`, `type`, `author`, `pin`, `date_pin`, `date`, `editable`) VALUES \n"

    # Parameters for the static fields
    # 1 = IndoPride
    # 2 = 日本の歌
    # 3 = Jowo Mletre
    # 4 = Worldwide
    favorite = "0"

    # Collecting each row of values
    values = []
    playlist_value = []

    for x in range(disc+1):
        # Buat URL API untuk folder
        # music_path itu adalah name folder lagunya.
        api_url = ""
        if "github.com/" in repo_url:
            api_url = f"https://api.github.com/repos/{owner}/{repo}/contents/{music_path[x]}"
        elif "https://bitbucket.org/" in repo_url:
            api_url = f"https://api.bitbucket.org/2.0/repositories/{owner}/{repo}/src/{branch}/{music_path[x]}?pagelen=100"
        elif "https://codeberg.org" in repo_url:
            api_url = f"https://codeberg.org/api/v1/repos/{owner}/{repo}/contents/{music_path[x]}"
        elif "https://gitlab.com/" in repo_url:
            api_url = f"https://gitlab.com/api/v4/projects/{owner}%2F{repo}/repository/tree?path={music_path[x]}&ref=main&per_page=100&page=1"
        elif "https://drive.google.com" in repo_url:
            # Untuk Google Drive, kita perlu menggunakan ID folder untuk mendapatkan konten
            api_url = f"https://www.googleapis.com/drive/v3/files?q='{music_path[x]}' in parents&fields=files(id,name,mimeType,webContentLink)&key={get_api_key()}"

        # Lakukan request ke API Git Repository
        response = requests.get(api_url)

        if response.status_code == 200:
            files = response.json()
            if "https://bitbucket.org/" in repo_url:
                files = files.get("values", [])
            elif "https://drive.google.com" in repo_url:
                # Untuk Google Drive, kita sudah mendapatkan file dari response sebelumnya
                files = response.json().get("files", [])
            index = 0
            album_disc = f"{album} Disc {x+1}"
            album_name = ''
            if disc > 0:
                album_name = album_disc
            else:
                album_name = album
            for file in files:
                get_true = file["type"] == file_type_value if "https://drive.google.com" not in repo_url else "audio/" in file["mimeType"]
                if get_true:
                    # Hapus ekstensi file untuk menampilkan nama file tanpa ekstensi
                    file_name_without_ext = os.path.splitext(
                        file[file_name_key])[0]
                    if "https://bitbucket.org/" in repo_url:
                        file_name_without_ext = file_name_without_ext.split("/")[-1]
                    title_song = ""

                    # cari artist berdasarkan nama file
                    if is_dynamic_artist:
                        artist = get_artist_from_title(
                            file_name_without_ext, list_title_artist[x])

                    # Cari judul berdasarkan index
                    if is_dynamic_title:
                        title_song = get_match_title_from_name(file_name_without_ext, list_index_title[x])
                        title_song = title_song.replace("\'", "\\'")

                    date_added = base_time + datetime.timedelta(seconds=index)
                    date_added_str = date_added.strftime("%Y-%m-%d %H:%M:%S")

                    index += 1
                    duration = generate_random_duration()

                    # Escape special characters in the values
                    artist = escape_sql_string(artist)
                    album = escape_sql_string(album)

                    # mendapatkan link git repo yang bisa di-stream
                    download_music_link = ""
                    if "https://drive.google.com" in repo_url:
                        # Untuk Google Drive, kita perlu menggunakan ID file untuk mendapatkan URL download
                        download_music_link = f"https://drive.google.com/file/d/{file['id']}/view?usp=drive_link"
                    elif "github.com/" in repo_url:
                        download_music_link = file['download_url']
                    elif "https://bitbucket.org/" in repo_url:
                        download_music_link = file['links']['self']['href']
                    elif "https://codeberg.org" in repo_url:
                        download_music_link = file['download_url']
                    elif "https://gitlab.com/" in repo_url:
                        disc_path = 'flac'
                        if disc > 0:
                            disc_path = f"{x+1}"
                        # download_music_link = f"https://gitlab.com/{owner}/{repo}/-/raw/{branch}/{folder_path}/{disc_path}/{file[file_name_key]}"
                        base_url = f"https://gitlab.com/{owner}/{repo}/-/raw/{branch}/{folder_path}/{disc_path}/"
                        file_path = f"{file[file_name_key]}"
                        # Encode hanya bagian path, bukan base
                        encoded_path = quote(file_path)  # spasi jadi %20
                        download_music_link = urljoin(base_url, encoded_path)

                    music_link = download_music_link

                    if "github.com/" in repo_url:
                        # mendapatkan link file yang versi encode
                        name_file_link_html = file['html_url']
                        # mengambil nama file dan ekstensi dari link html
                        name_link_encoded = name_file_link_html.split('/')[-1]
                        # memasukkan ke dalam link stream dengan nama file encode
                        music_link = download_music_link.rsplit(
                            '/', 1)[0] + '/' + name_link_encoded
                    if "https://gitlab.com/" not in repo_url:
                        # Encode ampersand
                        music_link = escape_ampersand(music_link)
                        # Encode other special characters
                        music_link = escape_url(music_link)
                        # Unquote untuk menghapus encoding berlebih
                        music_link = unquote(music_link)

                    cover_link = list_cover[(index-1) % len(list_cover)]
                    cover_link = escape_sql_string(cover_link)
                    if "github.com/" in repo_url:
                        cover_link = cover_link.replace(
                            "https://github.com/",
                            "https://raw.githubusercontent.com/"
                        ).replace(
                            "/blob/",
                            "/refs/heads/"
                        )
                    date_added_str = escape_sql_string(date_added_str)

                    final_title = ""
                    if is_dynamic_title == True:
                        final_title = title_song
                    else:
                        final_title = file_name_without_ext

                    values.append(
                        f"(NULL, '{category}', '{music_link}', '{final_title}', '{artist}', '{album_name}', '{duration}', '{cover_link}', '{favorite}', '{date_added_str}')"
                    )
                    if x == 0 and index == 1:
                        playlist_value.append(
                            f"(NULL, '{album}', '{cover_link}', 'album', '{album_artist}', 'false', NULL, NOW(), 'false')"
                        )
        else:
            print("Error:", response.status_code, response.json())

    # Joining all values into the final SQL INSERT statement
    insert_statement = insert_statement_template + \
            ",\n".join(values) + ";"
    playlist_insert_statement = playlist_insert_statement_template + \
            ",\n".join(playlist_value) + ";"
    print(insert_statement)
    print(playlist_insert_statement)


# Place URL here
repo_url = "https://drive.google.com/drive/folders/11BTGYt3btQWtkcLKZn4k3VoSAwILFdSc?usp=drive_link"

fetch_gitrepo_files(repo_url)

INSERT INTO music (id_music, category, link_gdrive, title, artist, album, time, cover, favorite, date_added) VALUES 
(NULL, '2', 'https://drive.google.com/file/d/1Ryz-iPpXFzuW_0J8T2e01z7EXTe2E2ac/view?usp=drive_link', 'Proud Corazón (Japanese Version / Karaoke Instrumental)', 'Cast - Coco', 'Coco (Original Motion Picture Soundtrack ／ Deluxe Edition) Disc 1', '03:17', 'https://drive.google.com/file/d/1vqF20LeYUuZ9_UOc8nxf4RDt44vGXHgW/view?usp=drive_link', '0', '2025-07-23 10:08:01'),
(NULL, '2', 'https://drive.google.com/file/d/1LHd7CD-AZF4Iwe3Ml66C-yg1LGbphb9b/view?usp=drive_link', 'Un Poco Loco (Japanese Version / Karaoke Instrumental]', 'Cast - Coco', 'Coco (Original Motion Picture Soundtrack ／ Deluxe Edition) Disc 1', '04:52', 'https://drive.google.com/file/d/1vqF20LeYUuZ9_UOc8nxf4RDt44vGXHgW/view?usp=drive_link', '0', '2025-07-23 10:08:02'),
(NULL, '2', 'https://drive.google.com/file/d/1-uuTB1jgf9Vq-JQe3Tfq-6CKQSc5_U_A/view?usp=drive_link', 'Remember Me (Reunion) [Japanese Version 